# Модуль 1. Бекенд (NestJS + PostgreSQL + SSE)

Документация: как устроен IoT-бекенд мониторинга температуры.

## 1. Что это

Бекенд:
1. принимает показания датчиков (`POST`);
2. пишет их в **PostgreSQL**;
3. отдаёт последние 50 записей (`GET`);
4. если `value <= -10` — пушит **алерт** всем подписчикам через **SSE**.

| Часть | Стек |
|-------|------|
| API | NestJS + TypeScript |
| БД | PostgreSQL + TypeORM |
| Валидация | class-validator |
| Конфиг | ConfigModule + `.env` |
| Алерты | RxJS Subject → `@Sse` |

## 2. Структура

```
src/
├── main.ts                 # CORS, ValidationPipe, listen
├── app.module.ts           # Config + TypeORM + модуль датчиков
└── temperature-sensors/
    ├── entities/…          # таблица temperature-sensors
    ├── dto/…               # что можно прислать в POST
    ├── temperature-alerts.service.ts
    ├── temperature-sensors.service.ts
    ├── temperature-sensors.controller.ts
    └── temperature-sensors.module.ts
```

## 3. Поток данных

```
POST JSON → Controller → ValidationPipe(DTO)
                ↓
     Service.create() → save() в PostgreSQL
                ↓
     AlertsService.checkAndEmitAlert()
                ↓
         value <= -10 ?  →  Subject.next()  →  SSE-клиенты
```

- `GET /temperature-sensors` — обычный JSON из БД.
- `GET /temperature-sensors/alerts` — долгий HTTP-поток (`text/event-stream`), сервер сам пушит события.

## 4. Сущность (таблица `temperature-sensors`)

| Поле | Тип | Зачем |
|------|-----|--------|
| `id` | UUID PK | ключ |
| `sensorName` | varchar(200) | имя датчика |
| `value` | double precision | температура |
| `unit` | varchar(20) | единица (`C` и т.д.) |
| `timestamp` | timestamptz, default NOW() | время замера |
| `createdAt` | CreateDateColumn | когда записали в БД; по нему сортируем топ-50 |

## 5. API

База: `http://localhost:3000`

### `POST /temperature-sensors`
Тело:
```json
{ "sensorName": "sensor-1", "value": -15.5, "unit": "C" }
```
Сохраняет запись. При `value <= -10` эмитит SSE-алерт.

### `GET /temperature-sensors`
Последние 50 записей, `createdAt DESC`.

### `GET /temperature-sensors/alerts` (SSE)
Поток алертов. Payload примерно:
```json
{
  "type": "critical_temperature",
  "message": "Critical temperature detected on sensor \"sensor-1\"",
  "sensorId": "…",
  "sensorName": "sensor-1",
  "value": -15,
  "unit": "C"
}
```

Порог: `CRITICAL_TEMPERATURE = -10` в `TemperatureAlertsService`.

## 6. SSE — как это работает

1. В `TemperatureAlertsService` живёт `Subject<MessageEvent>`.
2. Контроллер: `@Sse('alerts')` → отдаёт `getAlertStream()` (Observable).
3. После каждого `save()` — `checkAndEmitAlert(saved)`.
4. Если холодно (`<= -10`) → `subject.next({ data: … })`.
5. Все открытые клиенты на `/alerts` получают событие.

**Не** polling. Сервер пушит только при критическом замере.  
`Subject` в памяти процесса: рестарт Nest = все SSE отвалились, истории алертов нет.

## 7. PostgreSQL и `.env`

```
DB_HOST=localhost
DB_PORT=5432
DB_USERNAME=postgres
DB_PASSWORD=postgres
DB_NAME=iot_monitoring
PORT=3000
```

- Локально (`localhost`) — без SSL.
- Render / удалённый хост — SSL с `rejectUnauthorized: false`.
- `synchronize: true` — TypeORM сам поднимает схему из entity (для лабы норм).

## 8. Запуск и проверка

```bash
npm install
npm run start:dev
```

```bash
# обычная — алерта нет
curl -X POST http://localhost:3000/temperature-sensors \
  -H "Content-Type: application/json" \
  -d '{"sensorName":"s1","value":22,"unit":"C"}'

# критическая — алерт в SSE
curl -X POST http://localhost:3000/temperature-sensors \
  -H "Content-Type: application/json" \
  -d '{"sensorName":"s1","value":-15,"unit":"C"}'

curl http://localhost:3000/temperature-sensors
curl -N http://localhost:3000/temperature-sensors/alerts
```

Фронт:
```js
const es = new EventSource('http://localhost:3000/temperature-sensors/alerts');
es.onmessage = (e) => console.log(JSON.parse(e.data));
```

## 9. Кто за что отвечает

| Файл | Роль |
|------|------|
| entity | строка в БД |
| dto | контракт входа POST |
| sensors.service | save + findLatest + вызов алертов |
| alerts.service | Subject + порог -10 |
| controller | HTTP: POST / GET / SSE |
| module | склеивает всё |
| app.module | env + TypeORM |
| main.ts | CORS + ValidationPipe |

Полный текст теми же словами: `docs/MODULE1_BACKEND.md`.